# Mekong Flood Intelligence — Level 3 training on Colab

Runs the remaining Level 3 configs exactly as they run locally (`scripts/level3_train.py`), so results are comparable and the frozen protocol still holds.

**Before running:** upload `data/colab/chip_cache.zip` (0.75 GB, made locally) to your Google Drive at `MyDrive/mfi/chip_cache.zip`.

Runtime → Change runtime type → **GPU** (T4 is fine).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!git clone --quiet https://github.com/SoSereysokbotra/mekong-flood-analysis.git
%cd mekong-flood-analysis
!git log --oneline | grep -c 'Freeze evaluation plan v1'   # must print 1: the training script checks this
!pip install -q rasterio pystac-client planetary-computer segmentation-models-pytorch pyyaml scikit-image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data/interim results/level3 results/level3/_logs
!unzip -q /content/drive/MyDrive/mfi/chip_cache.zip -d /tmp/cc
!mv /tmp/cc/chip_cache data/interim/chip_cache
!cp /tmp/cc/_norm_stats.json results/level3/_norm_stats.json
!ls data/interim/chip_cache | wc -l   # 446

## Train the remaining configs

`unet_vvvh_ce` and `unet_vv_ce` were finished locally and are skipped (their `metrics_valid.json` is in the repo). Each config takes ~10–15 min on a T4. Test is **never** scored here — that happens locally after `--select`.

In [ ]:
import os, subprocess
ORDER = ['unet_vvvh_ce', 'unet_vv_ce', 'unet_vvvh_cropw', 'unet_vvvh_dice_ce', 'unet_vvvh_focal', 'unet_vvvh_slope', 'unet_vvvh_noaug', 'unet_r18_vvvh']
for c in ORDER:
    if os.path.exists(f'results/level3/{c}/metrics_valid.json'):
        print('skip', c, '(done)'); continue
    print('=== training', c)
    with open(f'results/level3/_logs/{c}.log', 'w') as log:
        r = subprocess.run(['python', 'scripts/level3_train.py', f'configs/level3/{c}.yaml'], stdout=log, stderr=subprocess.STDOUT)
    !grep -E "^ep |best epoch|Error|Traceback" results/level3/_logs/{c}.log | tail -4
print('ALL DONE')

## Package results for the local repo

Everything the local `scripts/merge_colab_results.py` needs: per-run metrics, configs, histories, checkpoints, logs, and the experiment-log rows appended here. Predictions on valid are left out (they are regenerated locally if needed).

In [ ]:
!mkdir -p /content/drive/MyDrive/mfi
!cd results && zip -q -r /content/drive/MyDrive/mfi/level3_colab_results.zip experiment_log.csv level3 -x 'level3/*/pred_valid/*'
!ls -la /content/drive/MyDrive/mfi/
print('Download level3_colab_results.zip to data/colab/ locally, then run: python scripts/merge_colab_results.py')